In [2]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [4]:
load_dotenv(override=True)
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama') 

In [5]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print("Pushover user found and looks good")
else:
    print("Pushover user not found")

if pushover_token:
    print("Pushover token found and looks good")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [8]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return "OK"

def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return "OK"

In [2]:
from dataclasses import dataclass


record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"},
            "name": {"type": "string", "description": "The user's name, if they provided it"},
            "notes": {"type": "string", "description": "Any additional info about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question that couldn't be answered"},
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

@dataclass
class Tool:
    name: str
    description: str
    function: callable

tools = [Tool("record_user_details_json", "Use this tool to record that a user is interested in being in touch and provided an email address", record_user_details_json),
        Tool("record_unknown_question_json", "Always use this tool to record any question that couldn't be answered as you didn't know the answer", record_unknown_question_json)]

tool_map = {tool.name: tool.function for tool in tools}

In [ ]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        func = tool_map.get(tool_name)
        result = func(**arguments) if func else "No tool found"
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results